# Statistics for Data Analysts/Engineers — Module 9: Chi-Square Tests

The t-tests from Module 8 compare the MEANS of continuous variables. But what about
**categorical** variables -- city, product category, campaign group, order status? The
**chi-square** (χ²) test family answers two related questions: whether two categorical
features are related (a **test of independence**), and whether an observed category
distribution matches an expected one (a **goodness-of-fit test**).

## Table of contents
1. [Contingency tables (crosstab)](#sec1)
2. [Chi-square test of independence](#sec2)
3. [Test assumptions and small cell counts](#sec3)
4. [Chi-square goodness-of-fit test](#sec4)
5. [Effect size: Cramér's V](#sec5)
6. [Summary and exercises](#sec6)


Every module in this course starts by generating the same synthetic dataset -- thanks to a fixed random seed (`np.random.default_rng(42)`), the numbers always come out identical, so you can open any module independently of the others, even after restarting the kernel.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

city = rng.choice(
    ["New York", "Chicago", "Austin", "Seattle", "Denver"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["North", "South", "East", "West"], size=n)
age = rng.integers(18, 70, size=n)

# monthly spend: right-skewed distribution (typical for spending data)
monthly_spend = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satisfaction correlated with spend + a region effect (deliberately different) + random
# noise, clipped to a 1-10 scale -- this gives Module 10 (ANOVA) a real between-group gap
region_effect = {"North": 0.0, "South": -0.5, "East": 0.3, "West": 0.8}
satisfaction = np.clip(
    monthly_spend / 45
    + np.array([region_effect[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# A/B campaign group -- with a REAL injected effect on conversion (group B better)
group = rng.choice(["A", "B"], size=n)
conversion_prob = np.where(group == "A", 0.10, 0.20)
converted = rng.binomial(1, conversion_prob)

# spend before/after a loyalty-program rollout (paired data)
spend_before = rng.normal(220, 40, size=n).round(2)
program_effect = rng.normal(15, 10, size=n)
spend_after = (spend_before + program_effect).round(2)

customers = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "city": city,
    "region": region,
    "age": age,
    "monthly_spend": monthly_spend,
    "satisfaction": satisfaction,
    "campaign_group": group,
    "converted": converted,
    "spend_before": spend_before,
    "spend_after": spend_after,
})
customers.head()


<a id="sec1"></a>
## 1. Contingency tables (crosstab)

A **contingency table** (a cross-tab) tallies counts for every combination of two
categorical variables. It's the starting point for every chi-square independence test.
Let's build one: campaign group (A/B) x conversion (yes/no) -- exactly the data we
analyzed differently in Modules 3 and 7.

In [ ]:
table = pd.crosstab(customers["campaign_group"], customers["converted"])
table


`pd.crosstab` with `normalize="index"` shows row-wise proportions right away -- handy
for a quick visual comparison before running a formal test:

In [ ]:
pd.crosstab(customers["campaign_group"], customers["converted"], normalize="index")


<a id="sec2"></a>
## 2. Chi-square test of independence

The **chi-square test of independence** checks whether two categorical variables are
independent of each other (Module 3!).

- H0: campaign group and conversion are **independent** (no relationship).
- H1: campaign group and conversion are **dependent** (a relationship exists).

The test statistic compares the observed counts with the counts we'd expect IF the
variables really were independent (i.e. if the conversion rate were identical in both
groups).

In [ ]:
from scipy import stats

chi2, p_value, df, expected = stats.chi2_contingency(table)

print(f"Chi2 statistic: {chi2:.3f}")
print(f"Degrees of freedom: {df}")
print(f"P-value: {p_value:.4f}")
print("Expected counts (if H0 were true):")
print(expected)


In [ ]:
alpha = 0.05
if p_value <= alpha:
    print("Reject H0 -- campaign group and conversion are NOT independent")
else:
    print("Fail to reject H0 -- no evidence of dependence")


> 📊 **Chi-square formalizes what we saw informally**
>
> This is exactly the same A/B effect we checked with conditional probability in Module 3, compared with proportion confidence intervals in Module 6, and tested with a one-proportion z-test in Module 7. The chi-square test of independence is the natural tool when you have TWO groups to compare on a categorical variable -- and it generalizes easily to MORE than two groups at once (e.g. conversion by city, not just by the two A/B groups).

<a id="sec3"></a>
## 3. Test assumptions and small cell counts

The chi-square test relies on an approximation and requires the **expected count**
(the `expected` values above) in every table cell to be **at least 5**. With smaller
expected counts, the chi-square approximation becomes unreliable.

In [ ]:
min_expected = expected.min()
print(f"Smallest expected count in the table: {min_expected:.2f}")
print("Assumption met" if min_expected >= 5 else "WARNING: assumption NOT met")


> ⚠️ **When counts are too small: Fisher's exact test**
>
> For small tables (especially `2x2`) with low expected counts, a better choice is **Fisher's exact test** (`stats.fisher_exact`) -- it computes an exact probability instead of relying on the chi-square approximation. It's computationally slower for large tables, but exact even with very small counts.

In [ ]:
# Fisher's exact test only works on 2x2 tables
odds_ratio, p_fisher = stats.fisher_exact(table)
print(f"Odds ratio: {odds_ratio:.3f}, p-value (Fisher): {p_fisher:.4f}")
print(f"(for comparison, chi-square p-value: {p_value:.4f})")


<a id="sec4"></a>
## 4. Chi-square goodness-of-fit test

The **goodness-of-fit** test checks whether the observed distribution of ONE categorical
variable matches an assumed, expected distribution. Recall from the data-generating code:
the `city` column was drawn with specific proportions (New York 30%, Chicago 25%,
Austin/Seattle/Denver 15% each). Let's check whether the observed sample actually matches
those assumed proportions.

- H0: the distribution of cities in the sample matches the assumed proportions.
- H1: the distribution of cities differs from the assumed proportions.

In [ ]:
assumed_proportions = {"New York": 0.30, "Chicago": 0.25, "Austin": 0.15, "Seattle": 0.15, "Denver": 0.15}

observed = customers["city"].value_counts().reindex(assumed_proportions.keys())
n = len(customers)
expected_gof = pd.Series(assumed_proportions) * n

print("Observed:")
print(observed)
print("\nExpected under H0:")
print(expected_gof)

chi2_gof, p_value_gof = stats.chisquare(f_obs=observed, f_exp=expected_gof)
print(f"\nchi2 = {chi2_gof:.3f}, p-value = {p_value_gof:.4f}")


Since we actually generated the data from those exact proportions, expect a large
p-value (no grounds to reject H0) -- the observed differences are just random sampling
noise around the true proportions in a sample of 400 customers.

<a id="sec5"></a>
## 5. Effect size: Cramér's V

Similar to Cohen's d for t-tests (Module 8), **Cramér's V** measures the STRENGTH of the
association between two categorical variables, independent of sample size (and therefore
independent of whether the test came out "significant"):

```
V = sqrt(chi2 / (n * (min(k, r) - 1)))
```

where `k` and `r` are the number of columns and rows in the table. `V` falls in the range
`[0, 1]` -- `0` means no association, `1` means a perfect association. Rough thresholds
(as conventional as Cohen's d's): `0.1` small, `0.3` medium, `0.5`+ large effect (for a
2x2 table).

In [ ]:
def cramers_v(contingency_table):
    chi2_v, _, _, _ = stats.chi2_contingency(contingency_table)
    n_v = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    return (chi2_v / (n_v * min_dim)) ** 0.5


v = cramers_v(table)
print(f"Cramer's V (campaign group x conversion): {v:.3f}")


<a id="sec6"></a>
## 6. Summary and exercises

In this module we covered:
- contingency tables (`pd.crosstab`) as the starting point for analyzing categorical
  variables,
- the chi-square test of independence (`stats.chi2_contingency`),
- the minimum expected-cell-count assumption and its alternative -- Fisher's exact test,
- the chi-square goodness-of-fit test (`stats.chisquare`) for a single variable,
- Cramér's V as a measure of association strength, independent of sample size.

> 📝 **Exercise 1: Region and order status**
>
> Build a contingency table of `region` x `converted` for all 4 regions (not just the A/B campaign groups) and run a chi-square test of independence. Does region seem to be related to conversion?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
region_table = pd.crosstab(customers["region"], customers["converted"])
chi2_r, p_r, df_r, expected_r = stats.chi2_contingency(region_table)
print(region_table)
print(f"chi2={chi2_r:.3f}, df={df_r}, p={p_r:.4f}")
```

Conversion in our data depends ONLY on campaign group, not on region -- expect a p-value above 0.05 (i.e. no significance), though it may come out fairly close to the threshold -- that's normal, random sample noise sometimes nudges close to the significance line even without a real effect.
</details>

> 📝 **Exercise 2: Checking expected counts**
>
> For the table from exercise 1, check the minimum expected cell count. Is the `>= 5` assumption met for this table?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
print(expected_r.min())
print("Assumption met" if expected_r.min() >= 5 else "Assumption NOT met")
```
</details>

> 📝 **Exercise 3: Goodness-of-fit for the campaign group split**
>
> The data-generating code draws `campaign_group` with equal 50/50 proportions. Run a chi-square goodness-of-fit test checking whether the observed A/B split in our sample matches 50/50 proportions.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
observed_groups = customers["campaign_group"].value_counts().reindex(["A", "B"])
expected_groups = pd.Series([0.5, 0.5], index=["A", "B"]) * len(customers)
chi2_g, p_g = stats.chisquare(f_obs=observed_groups, f_exp=expected_groups)
print(f"chi2={chi2_g:.3f}, p={p_g:.4f}")
```
</details>

> 📝 **Exercise 4: Cramér's V for regions**
>
> Compute Cramér's V for the `region` x `converted` table from exercise 1, using the `cramers_v` function from section 5. Compare it with Cramér's V for campaign group x conversion from section 5 -- which association is stronger?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
v_regions = cramers_v(region_table)
print(f"Cramer's V (region x conversion): {v_regions:.3f}")
print(f"Cramer's V (campaign group x conversion): {v:.3f}")
```

Surprisingly, both V values can come out quite close, even though the region test (exercise 1) was NOT statistically significant while the campaign group test (section 2) was. Important lesson: Cramer's V depends only on chi2/n/table dimensions, not directly on degrees of freedom the way the p-value does -- with a larger table (4 regions), plain sampling noise more easily produces a moderate V even without a real association. Effect size and significance always need to be read together, not separately.
</details>

> 🔥 **Challenge: a simulation-based check on the independence test**
>
> Verify the chi-square independence test from section 2 with a permutation method: shuffle the `converted` column randomly 10,000 times (independently of `campaign_group`), computing the chi-square statistic for the shuffled table each time (`stats.chi2_contingency(pd.crosstab(customers['campaign_group'], shuffled_conversion))[0]`). Compute the fraction of permutations where the permuted statistic >= the original statistic -- that's your permutation p-value. Compare it with the result from section 2.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import numpy as np

rng = np.random.default_rng(4)
num_permutations = 2000
extreme_count = 0

for _ in range(num_permutations):
    shuffled = rng.permutation(customers["converted"].values)
    sim_table = pd.crosstab(customers["campaign_group"], shuffled)
    chi2_sim, _, _, _ = stats.chi2_contingency(sim_table)
    if chi2_sim >= chi2:
        extreme_count += 1

permutation_p = extreme_count / num_permutations
print(f"P-value (permutation): {permutation_p:.4f}")
print(f"P-value (chi2_contingency, section 2): {p_value:.4f}")
```

Shuffling deliberately destroys any association between the columns (since we permute one independently of the other), so it simulates exactly what H0 assumes -- the similarity to the formal test's result is extra confirmation that it's working correctly.
</details>

## What's next?

T-tests (Module 8) compare two means, and chi-square tests (this module) compare
categories. What if we want to compare means across **more than two** groups at once
(e.g. satisfaction across all four regions)? In **Module 10** we'll learn **ANOVA** --
analysis of variance, designed exactly for this task -- and formally test the same
regional difference we saw visually back in Module 2.